In [ ]:

import nglview as nv
import MDAnalysis as mda
from MDAnalysis.analysis import distances
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import numpy as np
import parmed
from parmed import load_file
import math
import datetime
from datetime import datetime
import prolif as plf
import matplotlib.patches as mpatches


In [ ]:
total_conts_df = pd.DataFrame()

for i in np.arange(0,45).tolist():

    temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/hnrnpa1/75mgml/plf_files/{str(i)}_conts.csv',
                          index_col=0, low_memory=False)

    temp_df = temp_df.replace({'True':1, 'False':0})

    conts_per_res_per_frame = temp_df.iloc[3:].values.sum() / len(temp_df.iloc[3:].index) / 10 

    total_conts_df[f'{temp_df.columns[0][:3]} {str(i)}'] = [conts_per_res_per_frame]
    

In [ ]:
total_conts_df.T.plot(kind='bar', figsize=[12,7]);
plt.ylabel('Contacts per residue per frame')
plt.xlabel('hnRNP-A1 Residue')
plt.tight_layout();
plt.savefig('/Users/williambrown/Desktop/hnrnpA1_total_contacts.pdf', dpi=500, transparent=True)

In [ ]:

for i in ['equal','charged','aromatic','polar','nonpolar']:
    u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/{i}/output.pdb')
    prot = u.select_atoms('protein')
    n_residues = prot.residues.n_residues
    n_res_dict[f'3{i[0].upper()}'] = n_residues
    

In [ ]:
for i in ['equal','charged','aromatic','polar','nonpolar']:
    u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/{i}/output.pdb')
    prot = u.select_atoms('protein')
    n_residues = prot.residues.n_residues
    n_res_dict[f'9{i[0].upper()}'] = n_residues

In [ ]:
for i in ['2_close','2_spread','3_close','3_spread']:
    if i == '2_close':
        name = 'GGGXGXGGG'
    elif i == '2_spread':
        name = 'GXGGGGGXG'
    elif i == '3_close':
        name = 'GGGXXXGGG'
    else:
        name = 'GXGGXGGXG'
    u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/{i}/output.pdb')
    prot = u.select_atoms('protein')
    n_residues = prot.residues.n_residues
    n_res_dict[f'{name}'] = n_residues

In [ ]:
n_res_dict

In [ ]:
direct_conts = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_9_conts_timeseries.csv', index_col=0, low_memory=False)

close_spread = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/close_spread_conts_timeseries.csv', index_col=0, low_memory=False)

all_wb_conts = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/tot_wb_conts.csv', index_col=0, low_memory=False)

water_content = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/good_water_contents/tot_water_content.csv', index_col=0)

direct_conts = pd.concat([direct_conts, close_spread], axis=1)

direct_conts = direct_conts.rename(columns={'3_E':'3E', '3_C':'3C', '3_A':'3A', '3_P':'3P',
       '3_N':'3N', '9_E':'9E', '9_C':'9C', '9_A':'9A',
       '9_P':'9P', '9_N':'9N', '2_close':'GGGXGXGGG','2_spread':'GXGGGGGXG', '3_close':'GGGXXXGGG',
       '3_spread':'GXGGXGGXG'})

all_conts = pd.DataFrame()
for i,j in n_res_dict.items():
    for k in direct_conts.columns:
        if i == k:
            all_conts.loc['Direct',f'{k}'] = direct_conts[k].mean() / n_res_dict[k]

for i,j in n_res_dict.items():
    for k in all_wb_conts.columns:
        if i == k:
            all_conts.loc['Water-Bridged',f'{k}'] = all_wb_conts[k].mean()

all_conts = all_conts.T
all_conts['Water Content'] = water_content.mean(axis=0).values


In [ ]:
all_conts = all_conts.T
all_conts = all_conts.rename(columns={'3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P',
       '3N':'3N', '9E':'9E', '9C':'9C', '9A':'9A',
       '9P':'9P', '9N':'9N', 'GGGXGXGGG':'X2','GXGGGGGXG':'X2S', 'GGGXXXGGG':'X3',
       'GXGGXGGXG':'X3S'})
all_conts = all_conts.T

all_conts.loc['X3S','Direct'] += 0.1

# Create a 3D scatter plot
plt.style.use("default")
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111)

# Plot the data points
ax.scatter(all_conts['Water Content'], all_conts['Direct'], c='r', marker='o', alpha=0)
                                                    
for i, let in enumerate(all_conts.index):
    if let in ['3E','3C','3A','3P','3N']:
        color = '#2CA02C'
    elif let in ['9E','9C','9A','9P','9N']:
        color = '#7E4AB8'
    else:
        color = '#FF7F0E'
    ax.text(all_conts['Water Content'][i], all_conts['Direct'][i], all_conts.index[i], fontsize=20, color = color)
                                                    
ax.set_ylabel('Protein-Protein Contacts Per Residue', fontsize=14)
ax.set_xlabel('Water:Peptide Mass Ratio', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=12, pad=0)
plt.ylim(6.2,7.8)
plt.xlim(0.2,2.9)

# name = system.capitalize()

# ax.set_title(f'{name} Stoichiometry')

plt.tight_layout()

plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/pp_wc_scatter.png', dpi=500, transparent=True)
plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/pp_wc_scatter.pdf', dpi=500, transparent=True)



In [ ]:
# Example: Change the bar colors for both primary and secondary y-axis
ax = all_conts.T.drop(columns=['Water Content']).plot(
    kind='bar',
    secondary_y='Water-Bridged',
    rot=45,
    color=['firebrick', 'teal'],
    figsize=[8,5]
    )

# Set custom y-limits for the primary y-axis
ax.set_ylim(6.2, 7.6)  # Replace with desired limits for the primary axis
ax.right_ax.set_ylim(0.3, 0.55)  # Replace with desired limits for the secondary axis

# Add labels for both y-axes
ax.set_ylabel("Protein-Protein Contacts Per Residue", fontsize=14)  # Replace with your primary axis label
ax.right_ax.set_ylabel("Water-Bridged Contacts Per Residue", fontsize=14)  # Replace with your secondary axis label
ax.set_xlabel('System', fontsize=14)

# Set font size for x-ticks and y-ticks
ax.tick_params(axis='x', labelsize=12)  # Set x-tick font size
ax.tick_params(axis='y', labelsize=12)  # Set y-tick font size for primary y-axis
ax.right_ax.tick_params(axis='y', labelsize=12)  # Set y-tick font size for secondary y-axis

plt.tight_layout()

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/direct_vs_wb_conts.png', dpi=500,transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/direct_vs_wb_conts.pdf', dpi=500,transparent=True)

plt.show()

In [ ]:
# Example: Change the bar colors for both primary and secondary y-axis
ax = all_conts.T.drop(columns=['Water-Bridged']).plot(
    kind='bar',
    secondary_y='Water Content',
    rot=45,
    color=['firebrick', '#126806'],
    figsize=[8,5]
    )

# Set custom y-limits for the primary y-axis
ax.set_ylim(6.2, 7.6)  # Replace with desired limits for the primary axis
# ax.right_ax.set_ylim(0.3, 0.55)  # Replace with desired limits for the secondary axis

# Add labels for both y-axes
ax.set_ylabel("Protein-Protein Contacts Per Residue", fontsize=14)  # Replace with your primary axis label
ax.right_ax.set_ylabel("Water:Peptide Mass Ratio", fontsize=14)  # Replace with your secondary axis label
ax.set_xlabel('System', fontsize=14)

# Set font size for x-ticks and y-ticks
ax.tick_params(axis='x', labelsize=12)  # Set x-tick font size
ax.tick_params(axis='y', labelsize=12)  # Set y-tick font size for primary y-axis
ax.right_ax.tick_params(axis='y', labelsize=12)  # Set y-tick font size for secondary y-axis

plt.tight_layout()

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/direct_vs_content_conts.png', dpi=500,transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/direct_vs_content_conts.pdf', dpi=500,transparent=True)

plt.show()

In [ ]:
water_content

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# water_content = water_content.rename(columns={'3E':'3E', '3C':'3C', '3A':'3A', '3P':'3P',
#        '3N':'3N', '9E':'9E', '9C':'9C', '9A':'9A',
#        '9P':'9P', '9N':'9N', '2close':'X2','GXGGGGGXG':'X2S', 'GGGXXXGGG':'X3',
#        'GXGGXGGXG':'X3S'})

# Bootstrapping Function
def bootstrap(data, n_iterations=10000, confidence_level=0.95):
    boot_means = np.array([
        np.mean(np.random.choice(data, size=len(data), replace=True))
        for _ in range(n_iterations)
    ])
    lower_bound = np.percentile(boot_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(boot_means, (1 + confidence_level) / 2 * 100)
    return np.mean(boot_means), lower_bound, upper_bound

# Window Size for Analysis
window_size = 10  # Adjust as needed

# Store Results for Each Column
results = {}

for column in water_content.columns[1:]:  # Skip the 'time' column
    column_results = []
    for i in range(0, len(water_content), window_size):
        window_data = water_content[column].iloc[i:i + window_size]
        if len(window_data) > 0:  # Ensure the window has data
            mean, lower, upper = bootstrap(window_data)
            column_results.append({
                'start_time': water_content['time'].iloc[i],
                'end_time': water_content['time'].iloc[min(i + window_size - 1, len(water_content) - 1)],
                'mean_time': ((water_content['time'].iloc[i])+(water_content['time'].iloc[min(i + window_size - 1, len(water_content) - 1)]))/2,
                'mean': mean,
                'lower': lower,
                'upper': upper
            })
    results[column] = pd.DataFrame(column_results)

# Plot Results for All Columns
plt.figure(figsize=(7, 5))

colors = [
    # Set 1: Blues (lightest to darkest)
    "#A6CEE3", "#6BAED6", "#3182BD", "#1F77B4", "#08519C",
    # Set 2: Greens (lightest to darkest)
    "#A1D99B", "#74C476", "#31A354", "#2CA02C", "#006D2C",
    # Set 3: Oranges (lightest to darkest)
    "#FDD0A2", "#FDAE6B", "#FF7F0E", "#E6550D"]
loop_num=0
for column, result_df in results.items():
    color = colors[loop_num]
    if column[1] != '_':
        plt.plot(result_df['mean_time'], result_df['mean'], label=f'{column[0]}{column[5].capitalize()}', marker='o', color=color)
        plt.fill_between(result_df['mean_time'], result_df['lower'], result_df['upper'], alpha=0.3, color=color)#, label=f'{column} 95% CI')
        loop_num += 1
    else:
        if column == '2Close':
            label = 'X2'
        elif column == '2Spread':
            label = 'X2S'
        elif column == '3Close':
            label = 'X3'
        else:
            label = 'X3S'
        plt.plot(result_df['mean_time'], result_df['mean'], label=f'{label}', marker='o', color=color)
        plt.fill_between(result_df['mean_time'], result_df['lower'], result_df['upper'], alpha=0.3, color=color)#, label=f'{column} 95% CI')
        loop_num += 1

plt.xlabel('Time (ns)', fontsize=14)
plt.ylabel('Water:Peptide Mass Ratio', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.ylim(0,3)
plt.xlim(0,5000)
# plt.title('Bootstrapping Convergence Analysis for All Columns')
plt.legend(ncols=3)

# plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/convergence/pngs/mass_ratio_convergence.png', dpi=500, transparent=True)
# plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/convergence/pdfs/mass_ratio_convergence.pdf', dpi=500, transparent=True)

plt.show()



In [ ]:
pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/').plot()

In [ ]:
dir_conts = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3a_9c_9p_conts_timeseries.csv', index_col=0)
# dir_conts = dir_conts.dropna(axis=0)

water_content = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/new_total_water_df.csv', index_col=0)
water_content.index = water_content.index * (5000/np.max(water_contenat.index))

wb_df = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/wb_conts.csv', index_col=0)


In [ ]:
dir_conts

In [ ]:
plt.plot(dir_conts)

In [ ]:
water_content

In [ ]:
# Define a moving average function
def moving_average(data, window_size):
    return np.convolve(data, np.ones(window_size) / window_size, mode='valid')

# Create figure
fig, ax = plt.subplots(figsize=[7, 5])

# Define the x-axis (0 to 5000 evenly spaced)
x = np.linspace(0, 5000, int(water_content['9res_charged Mass Ratio'].sum()))

smoothed_wm = moving_average(water_content['3res_aromatic Mass Ratio'], window_size=10)
x_smoothed_wm = np.linspace(x[0], x[-1], len(smoothed_wm))
wm_interp = np.interp(x, x_smoothed_wm, smoothed_wm)
wm_normalized = (wm_interp - np.min(wm_interp)) / (np.max(wm_interp) - np.min(wm_interp))
ax.plot(x, wm_normalized, color='firebrick', label='3A')

smoothed_wm = moving_average(water_content['9res_charged Mass Ratio'], window_size=10)
x_smoothed_wm = np.linspace(x[0], x[-1], len(smoothed_wm))
wm_interp = np.interp(x, x_smoothed_wm, smoothed_wm)
wm_normalized = (wm_interp - np.min(wm_interp)) / (np.max(wm_interp) - np.min(wm_interp))
ax.plot(x, wm_normalized, color='olivedrab', label='9C')

smoothed_wm = moving_average(water_content['9res_polar Mass Ratio'], window_size=10)
x_smoothed_wm = np.linspace(x[0], x[-1], len(smoothed_wm))
wm_interp = np.interp(x, x_smoothed_wm, smoothed_wm)
wm_normalized = (wm_interp - np.min(wm_interp)) / (np.max(wm_interp) - np.min(wm_interp))
ax.plot(x, wm_normalized, color='goldenrod', label='9P')

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('Time (ns)', fontsize=14)
plt.ylabel('Water:Peptide Mass Ratio', fontsize=14)

plt.legend(fontsize=12)

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/water/low_water_contents.png', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/water/low_water_contents.pdf', dpi=500, transparent=True)

plt.show()


In [ ]:
residue_list = [5,10,15,20,25]
selection = " or ".join([f"resindex {num}" for num in residue_list])

In [ ]:
selection

In [ ]:
import MDAnalysis as mda
import numpy as np
from parmed import load_file 

topu = mda.Universe(load_file('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/2_close/output.top'),
                   '/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/2_close/traj.dcd')

pdbu = mda.Universe('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/2_close/output.pdb',
                   '/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/2_close/traj.dcd')

pdb_prot = pdbu.select_atoms('protein')
top_prot = topu.select_atoms('protein')

atom_list = pdb_prot.atoms.ids[top_prot.atoms.fragindices == int(np.max(top_prot.atoms.fragindices)/2)]
selection = " or ".join([f"id {num}" for num in atom_list])
chain_sel = pdbu.select_atoms(selection)
structure = chain_sel.residues.resnames


In [ ]:
def calist_listtostring(residue_list, structure):
    str1 = ""
    for ele in residue_list:
        for part in structure:
            if part != 'GLY':  # Replace 'VAL' with the current residue
                str1 += ele
            else:  # Add the 'GLY' parts directly
                str1 += part
    return str1

calist_listtostring(residue_list=['ALA','ARG','LEU'], structure=structure)


In [ ]:
pdbu.select_atoms()

In [ ]:
top_prot.atoms.fragindices

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define a moving average function
def moving_average(data, window_size):
    return np.convolve(data, np.ones(window_size) / window_size, mode='valid')

# Create figure
fig, ax = plt.subplots(figsize=[7, 5])

# Define the x-axis (0 to 5000 evenly spaced)
x = np.linspace(0, 5000, int(water_content['3res_aromatic Mass Ratio'].sum()))

# Smooth and interpolate "Water/Peptide Mass Ratio"
smoothed_wm = moving_average(water_content['3res_aromatic Mass Ratio'], window_size=10)
x_smoothed_wm = np.linspace(x[0], x[-1], len(smoothed_wm))
wm_interp = np.interp(x, x_smoothed_wm, smoothed_wm)
wm_normalized = (wm_interp - np.min(wm_interp)) / (np.max(wm_interp) - np.min(wm_interp))
plt.plot(x, wm_normalized, label='Water/Peptide Mass Ratio', color='firebrick')

# Smooth and interpolate "Water-Bridged Contacts"
wb_data = wb_df['3_res_aromatic'].iloc[10:]
smoothed_wb = moving_average(wb_data, window_size=10)
x_smoothed_wb = np.linspace(x[0], x[-1], len(smoothed_wb))
wb_interp = np.interp(x, x_smoothed_wb, smoothed_wb)
wb_normalized = (wb_interp - np.min(wb_interp)) / (np.max(wb_interp) - np.min(wb_interp))
plt.plot(x, wb_normalized, label='Water-Bridged Contacts', color='steelblue')

# Compute slopes (derivatives) of the curves using np.gradient
# Ensure that the x values for the gradient calculation match the lengths of the normalized arrays
wm_slope = np.gradient(wm_normalized, x)
wb_slope = np.gradient(wb_normalized, x)

# Find regions where slopes have opposite signs
opposite_signs = wm_slope * wb_slope < 0

# Shade regions with opposite slopes
for i in range(len(opposite_signs) - 1):
    if opposite_signs[i]:  # Check where slopes have opposite signs
        plt.fill_between(x[i:i+2], wm_normalized[i:i+2], wb_normalized[i:i+2], color='gray', alpha=0.6)

# Count the shaded points (where opposite_signs is True)
shaded_points = np.sum(opposite_signs)
total_points = len(opposite_signs)

# Calculate the percentage of shaded points
shaded_percentage = (shaded_points / total_points) * 100

# Display the percentage of shaded points on the plot
plt.text(0.95, 0.95, f'Shaded: {shaded_percentage:.2f}%', 
         transform=ax.transAxes, fontsize=12, ha='right', va='top')

# Customize the plot
# ax.yaxis.set_ticks([])                            # Remove ticks
# ax.tick_params(axis='y', which='both', length=0)  # Remove tick marks
plt.xticks(fontsize=12)
plt.xlabel('Time (ns)', fontsize=14)
plt.ylabel('Water-Bridged Contacts per Residue', fontsize=14)
plt.legend(fontsize=12)

# Get the original y-axis values for wb_df before normalization
original_wb_ticks = np.linspace(wb_data.min(), wb_data.max(), 5)  # Adjust the number of ticks as needed

# Normalize the tick values to match the normalized range (0 to 1)
normalized_wb_ticks = (original_wb_ticks - wb_data.min()) / (wb_data.max() - wb_data.min())

# Round original tick values to the nearest 0.1
rounded_wb_ticks = np.round(original_wb_ticks, 1)

# Set custom y-ticks and their labels for the original wb_df values
ax.set_yticks(normalized_wb_ticks)
ax.set_yticklabels([f'{tick:g}' for tick in rounded_wb_ticks], fontsize=12)  # Format to 2 decimal places

plt.savefig('/Users/williambrown/Desktop/3A_wb_wv_bicurve_shaded_wbyaxis.pdf', dpi=500, transparent=True)
plt.savefig('/Users/williambrown/Desktop/3A_wb_wv_bicurve_shaded_wbyaxis.png', dpi=500, transparent=True)

plt.show()


In [ ]:
import pandas as pd
import numpy as np

loop_num = 0

for system in ['9_res/equal/2_close','9_res/equal/2_spread','9_res/equal/3_close','9_res/equal/3_spread']:

    temp_df = pd.read_csv(f'./{system}/plf_files/all_conts.csv', index_col=0, low_memory=False)

    temp_df = temp_df.iloc[3:]

    temp_df = temp_df.replace({'True':1, 'False':0})

    if loop_num == 0:
        total_df = pd.DataFrame()
        total_df[f'{system[12:]}'] = temp_df.sum(axis=1).values
        loop_num += 1
    else:
        temp_df_2 = pd.DataFrame()
        temp_df_2[f'{system[12:]}'] = temp_df.sum(axis=1).values
        total_df = pd.concat([total_df, temp_df_2], axis=1)

    print(system)

total_df.to_csv('./close_spread_conts_timeseries.csv')

In [ ]:
# Normalize data
wm_normalized = (wm_interp[::2] - np.min(wm_interp[::2])) / (np.max(wm_interp[::2]) - np.min(wm_interp[::2]))
wb_normalized = (wb_interp[::2] - np.min(wb_interp[::2])) / (np.max(wb_interp[::2]) - np.min(wb_interp[::2]))

# Create matching x arrays
x_wm = np.linspace(x[0], x[-1], len(wm_normalized))
x_wb = np.linspace(x[0], x[-1], len(wb_normalized))

# Calculate derivatives
wm_derivative = np.gradient(wm_normalized, x_wm)
wb_derivative = np.gradient(wb_normalized, x_wb)

# Plot derivatives
plt.plot(x_wm, wm_derivative, label='Derivative of Water/Peptide Mass Ratio', color)
plt.plot(x_wb, wb_derivative, label='Derivative of Water-Bridged Contacts')

# Identify regions where slopes have opposite signs
opposite_signs = wm_derivative * wb_derivative < 0

# Group consecutive regions where slopes are opposite
shade_regions = []
start = None
for i, val in enumerate(opposite_signs):
    if val and start is None:  # Start of a new region
        start = i
    elif not val and start is not None:  # End of the current region
        shade_regions.append((start, i - 1))
        start = None
if start is not None:  # Handle case where the last region extends to the end
    shade_regions.append((start, len(opposite_signs) - 1))

# Shade grouped regions
for region in shade_regions:
    start_idx, end_idx = region
    plt.fill_between(
        x_wm[start_idx:end_idx + 1],  # x-range for shading
        wm_derivative[start_idx:end_idx + 1],  # Lower bound for shading
        wb_derivative[start_idx:end_idx + 1],  # Upper bound for shading
        color='gray',  # Shading color
        alpha=0.5,  # Transparency
        label='Opposite Slopes' if region == shade_regions[0] else ''  # Label only once
    )

opposite_percentage = np.sum(opposite_signs) / len(opposite_signs) * 100
plt.title(f'Derivatives with Opposite Slopes Highlighted ({opposite_percentage:.1f}% of the time)')

plt.legend()

# plt.savefig('/Users/williambrown/Desktop/9p_WB_WC_derivative_bicurve.pdf', dpi=500, transparent=True)
# plt.savefig('/Users/williambrown/Desktop/9p_WB_WC_derivative_bicurve.png', dpi=500, transparent=True)

plt.show()

## Chain Length & Composition

In [ ]:
for sys in ['3','9']: 
    
    res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
                'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']

    itype_df_dict = {}

    comp_list = ['equal','charged','aromatic','polar','nonpolar']

    prot_conts_per_frame = pd.DataFrame(index=comp_list, columns=res_list)

    for comp in comp_list:

        itype_df  = pd.DataFrame(columns=['VdWContact','HBDonor','Cationic','HBAcceptor','Hydrophobic','CationPi'], index=res_list)

        for res in res_list:
            
            if sys == '9':
                traj = 'traj.dcd'
            elif sys == '3':
                traj = 'trajectory.dcd'
            
            u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{sys}_res/{comp}/output.gro',
                            f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{sys}_res/{comp}/{traj}', guess_bonds=True)

            prot = u.select_atoms('protein and name CA')

            middle = math.ceil(Counter(prot.fragindices)[0]/2)

            res_sel = u.select_atoms(f'protein and resid {middle} and resname {res}')

            num_res = res_sel.residues.n_residues

            temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{sys}_res/{comp}/plf_files/{res}_plf.csv', index_col=0, low_memory=False)

            temp_dict = Counter(temp_df.iloc[1].values)

            for i,j in temp_dict.items():
                itype_df.loc[f'{res}',i] = j

            temp_df = temp_df.replace({'True':1})
            temp_df = temp_df.replace({'False':0})

            number = ((temp_df.iloc[3:].sum().sum()) / (len(temp_df.index[3:])) / (num_res))

            print(res, temp_df.iloc[3:].sum().sum(), len(temp_df.index[3:]), num_res)

            prot_conts_per_frame.loc[f'{comp}', f'{res}'] = number

        print(f'{sys}_{comp}')

        itype_df_dict[f'{comp}'] = itype_df
        

    for comp in comp_list:
        itype_df_dict[f'{comp}']['H-Bond'] = itype_df_dict[f'{comp}']['HBDonor'] + itype_df_dict[f'{comp}']['HBAcceptor']
        itype_df_dict[f'{comp}'] = itype_df_dict[f'{comp}'].drop(columns=['HBDonor','HBAcceptor'])

        itype_df_dict[f'{comp}']['Electrostatic'] = itype_df_dict[f'{comp}']['Anionic'] + itype_df_dict[f'{comp}']['Cationic']
        itype_df_dict[f'{comp}'] = itype_df_dict[f'{comp}'].drop(columns=['Anionic','Cationic'])

        itype_df_dict[f'{comp}']['Pi-Cation'] = itype_df_dict[f'{comp}']['PiCation'] + itype_df_dict[f'{comp}']['CationPi']
        itype_df_dict[f'{comp}'] = itype_df_dict[f'{comp}'].drop(columns=['PiCation','CationPi'])

        itype_df_dict[f'{comp}']['Pi-Stacking'] = itype_df_dict[f'{comp}']['FaceToFace'] + itype_df_dict[f'{comp}']['EdgeToFace'] + itype_df_dict[f'{comp}']['PiStacking']
        itype_df_dict[f'{comp}'] = itype_df_dict[f'{comp}'].drop(columns=['FaceToFace','EdgeToFace','PiStacking'])

        itype_df_dict[f'{comp}'].to_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/cont_counts/{sys}res_{comp}.csv')

    prot_conts_per_frame.to_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{sys}_res/total_conts_df.csv',)


In [ ]:
tot_conts_3res = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/total_conts_df.csv', index_col=0)

tot_conts_9res = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/total_conts_df.csv', index_col=0)

tot_conts_3res = tot_conts_3res.T

tot_conts_9res = tot_conts_9res.T

tot_conts_3res = tot_conts_3res / 2

tot_conts_9res = tot_conts_9res / 2


In [ ]:
tot_conts_9res.max()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import ternary
import mpltern
from mpl_toolkits.mplot3d import Axes3D

for system in ['equal','charged','aromatic','polar','nonpolar']:

    aromatic = ['F', 'Y', 'W']
    polar = ['C','T','Q','N','S']
    nonpolar = ['V','I','A','L','P','M']
    positive = ['R','K','H']
    negative = ['E','D']
    
    let_list = ['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V']
    tot_conts_3res['let_list'] = let_list
    
    # Create a 3D scatter plot
    plt.style.use("default")
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111)
    
    # Plot the data points
    ax.scatter(tot_conts_3res[f'{system}'], tot_conts_9res[f'{system}'], c='r', marker='o', alpha=0)
    
    aromatic = ['F', 'Y', 'W']
    polar = ['C','T','Q','N','S']
    nonpolar = ['V','I','A','L','P','M']
    positive = ['R','K','H']
    negative = ['E','D']
                                                        
    for i, let in enumerate(let_list):
        if let in positive:
            color = '#e6372d'
        elif let in negative:
            color = '#0D88E6'
        elif let in aromatic:
            color = '#fbbe28'
        elif let in polar:
            color = '#2d9c1c'
        elif let in nonpolar:
            color = '#8a2ce2'
        else:
            color = 'black'
        ax.text(tot_conts_3res[f'{system}'][i], tot_conts_9res[f'{system}'][i], tot_conts_3res['let_list'][i], fontsize=20, color = color)
                                                        
    ax.set_xlabel('3-Residue Chains', fontsize=14)
    ax.set_ylabel('9-Residue Chains', fontsize=14)
    ax.tick_params(axis='both', which='major', labelsize=12, pad=0)
    plt.xlim(2.5,7.6)
    plt.ylim(2.5,7.6)
    
    name = system.capitalize()
    
    ax.set_title(f'{name} Stoichiometry')
    
    plt.tight_layout()
    
    plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/conts/{name}_2d_scatter.png', dpi=500, transparent=True)
    plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/conts/{name}_2d_scatter.pdf', dpi=500, transparent=True)



In [ ]:
tot_conts_3res

In [ ]:
tot_conts_3res.T.boxplot(grid=False, figsize=[7,5])
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.ylabel('Average # Contacts', fontsize=14)
plt.xlabel('Residue', fontsize=14);
plt.ylim(2.5,7.75)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/conts/3res_cont_diffs.pdf', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/conts/3res_cont_diffs.png', dpi=500, transparent=True)
plt.show


In [ ]:
aromatic = ['PHE', 'TYR', 'TRP']
polar = ['CYS','THR','GLN','ASN','SER']
nonpolar = ['VAL','ILE','ALA','LEU','PRO','GLY','MET']
positive = ['ARG','LYS','HIS']
negative = ['GLU','ASP']

df = tot_conts_3res

ax = df.plot.bar(width=0.8, color=['grey','#b30039','#e3ba14','#7aaa3d', '#ac5cad'], figsize=(12,6));

# bars = ax.patches
# hatches = ''.join(h*len(df) for h in ' o   ')

# for i, (bar, hatch) in enumerate(zip(bars, hatches)):
#     if hatch != ' ':
#         ax.add_patch(
#             plt.Rectangle(
#                 (bar.get_x(), 0),
#                 bar.get_width(),
#                 bar.get_height(),
#                 facecolor=(0, 0, 0, 0.01),
#                 edgecolor='#0c7ace',
#                 hatch=hatch,
#                 linewidth=0,
#                 zorder=bar.get_zorder() + 1,
#                 rasterized=True  # Rasterize only the hatched overlay
#             )
#         )
#     else:
#         bar.set_edgecolor('none')  # Default edge color for non-hatched bars

plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)

# legend_handles = [
#     mpatches.Patch(facecolor='grey', edgecolor='none', label='Equal'),
#     mpatches.Patch(facecolor='#b30039', edgecolor='#0c7ace', hatch='o', label='Charged'),
#     mpatches.Patch(facecolor='#e3ba14', edgecolor='none', label='Aromatic'),
#     mpatches.Patch(facecolor='#7aaa3d', edgecolor='none', label='Polar'),
#     mpatches.Patch(facecolor='#ac5cad', edgecolor='none', label='Nonpolar')]
    

ax.legend(loc='upper right', ncol=5, fontsize=12)
plt.xlabel('Residue', fontsize=14)
plt.ylabel('# Contacts per Residue per Frame', fontsize=14)

for tick_label in ax.get_xticklabels():
    if str(tick_label)[-5:-2] in aromatic:
        tick_label.set_color('#e3ba14')
    elif str(tick_label)[-5:-2] in polar:
        tick_label.set_color('#7aaa3d')
    elif str(tick_label)[-5:-2] in positive:
        tick_label.set_color('#b30039')
    elif str(tick_label)[-5:-2] in negative:
        tick_label.set_color('#0c7ace')
    else:
        tick_label.set_color('#ac5cad')

plt.tight_layout();
        
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/3res_cluster_conts.pdf', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/3res_cluster_conts.png', dpi=500, transparent=True)


## 

## RNA-GXG

In [ ]:
res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
            'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']

type = 'nn'

if type == 'pp':
    str_len = 3
    num_chains = 100
elif type == 'nn':
    str_len = 1
    num_chains = 30
    
df = pd.DataFrame(columns = ['tot_conts'], index = res_list)
    
for residue in res_list:
    
    temp_df = pd.read_csv(f'/Volumes/potoyan-lab/will/rna_gxg/analysis_files/plf_files/nn_vdw/{residue}_nn_plf.csv', index_col=0, low_memory=True)

    start_string = temp_df.columns.tolist()[0]
    if start_string.__contains__('.'):
        str2 = '.'    
        idx2 = start_string.index(str2)
        start_res = f'{start_string[str_len:idx2]}'
    else:
        start_res = f'{start_string[str_len:]}'

    end_string = temp_df.columns.tolist()[-1]
    if end_string.__contains__('.'):
        str2 = '.'    
        idx2 = end_string.index(str2)
        end_res = f'{end_string[str_len:idx2]}'
    else:
        end_res = f'{end_string[str_len:]}'

    test_list = np.arange(int(start_res),int(end_res)+1).tolist()

    split_list = [test_list[i:i+3] for i in range(0,len(test_list),3)]

    for column in temp_df.columns.tolist():

        string = str(column)
        string2 = temp_df[column].iloc[0][str_len:]

        if string.__contains__('.'):
            str2 = '.'    
            idx2 = string.index(str2)
            res = f'{string[str_len:idx2]}'
        else:
            res = f'{string[str_len:]}'

        index = next((i for i, sublist in enumerate(split_list) if int(res) in sublist), -1)

        temp_list = split_list[index]

        if int(string2) in temp_list:
            temp_df = temp_df.drop(f'{column}', axis=1)
    
    temp_df = temp_df.replace({'True':1})
    temp_df = temp_df.replace({'False':0})
    
    temp_df = temp_df.iloc[3:].astype(int)
        
    df.loc[f'{residue}', 'tot_conts'] = temp_df.sum().sum() / len(temp_df.index[3:]) / num_chains

#     temp_df.to_csv(f'/Volumes/potoyan-lab/will/rna_gxg/analysis_files/plf_files/nn_vdw/{residue}_fix_plf.csv')
    print(residue, temp_df.sum().sum() / len(temp_df.index[3:]) / num_chains)

# df.to_csv(f'/Volumes/potoyan-lab/will/rna_gxg/analysis_files/plf_files/nn_vdw/all_diff_chain_nn.csv')

    
    

In [ ]:
ax = itype_df_dict['aromatic'].plot(kind='bar', stacked=True, figsize=[7,5])
plt.ylabel('Values')
plt.title('Stacked Bar Plot')
plt.legend(ncols=2, loc='upper left', fontsize='small')

# Display the plot
plt.show()

In [ ]:
prot_conts_per_frame.mean(axis=1)

In [ ]:
conts9_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/chain_length/9_res/rep_1/', index_col=0)
conts15_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/50mgml_100a/replica-2/central_res_hb_conts.csv', index_col=0)
conts21_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/50mgml_100a/replica-3/central_res_hb_conts.csv', index_col=0)

conts9_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/central_res_hb_conts.csv', index_col=0)
conts15_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-2/central_res_hb_conts.csv', index_col=0)
conts21_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-3/central_res_hb_conts.csv', index_col=0)

# conts250_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-1/central_res_hb_conts.csv', index_col=0)
# conts250_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-2/central_res_hb_conts.csv', index_col=0)
# conts250_3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-3/central_res_hb_conts.csv', index_col=0)

total_50 = (conts50_1 + conts50_2 + conts50_3) / 3
total_125 = (conts125_1 + conts125_2 + conts125_3) / 3
# total_250 = (conts250_1 + conts250_2 + conts250_3) / 3



In [ ]:
test = total_250 - total_125

In [ ]:
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
import matplotlib.colors as colors
import matplotlib.cbook as cbook
from matplotlib import cm

aromatic = ['PHE', 'TYR', 'TRP']
polar = ['CYS','THR','GLN','ASN','SER']
nonpolar = ['VAL','ILE','ALA','LEU','PRO','GLY','MET']
positive = ['ARG','LYS','HIS']
negative = ['GLU','ASP']

cmap = sns.diverging_palette(260, 15, as_cmap=True)
fig = plt.subplots(figsize=[7,5])

test = test[['ASP','GLU','ARG','LYS','HIS','TRP','TYR','PHE','ASN','GLN','SER','THR','CYS','PRO','VAL','MET','ILE','LEU','GLY']]
test = test.reindex(['ASP','GLU','ARG','LYS','HIS','TRP','TYR','PHE','ASN','GLN','SER','THR','CYS','PRO','VAL','MET','ILE','LEU','GLY'])

# ax = sns.heatmap(test, cmap='vlag', square=True, norm=LogNorm(), cbar_kws={'ticks':[0.003,0.01,0.03,0.1,0.3,0.8]})
ax = sns.heatmap(test, cmap=cmap, square=True, cbar=False)
pc = ax.pcolormesh(test, norm=colors.TwoSlopeNorm(vmin=test.min().min(),vcenter=0., vmax=-test.min().min()), cmap=cmap)
# plt.colorbar(pc, ax=ax)
cbar = plt.colorbar(pc, ax=ax, ticks=[-0.4,-0.3,-0.2,-0.1,0.0,0.1,0.2,0.3,0.4])

cbar.ax.tick_params(labelsize=13)

# colorbar = ax.collections[0].colorbar

# tick_size = 10

# Set the size of all ticks
# colorbar.ax.tick_params(labelsize=tick_size, length=4.1, width=1)

# colorbar.set_ticklabels(['0.003','0.01','0.03','0.1','0.3','0.8'])

ax.xaxis.tick_bottom()
ax.yaxis.tick_left()
ax.invert_yaxis()
# plt.title('Central Residue Hydrogen Bonding', fontsize=15)
#plt.ylabel('Residue', fontsize=13)
#plt.xlabel('Residue', fontsize=13)
plt.yticks(fontsize=13, rotation=30)
plt.xticks(fontsize=13, rotation=60)

for tick_label in ax.get_xticklabels():
    if str(tick_label)[-5:-2] in aromatic:
        tick_label.set_color('#e3ba14')
    elif str(tick_label)[-5:-2] in polar:
        tick_label.set_color('#7aaa3d')
    elif str(tick_label)[-5:-2] in positive:
        tick_label.set_color('#b30039')
    elif str(tick_label)[-5:-2] in negative:
        tick_label.set_color('#0c7ace')
    else:
        tick_label.set_color('#ac5cad')

for tick_label in ax.get_yticklabels():
    if str(tick_label)[-5:-2] in aromatic:
        tick_label.set_color('#e3ba14')
    elif str(tick_label)[-5:-2] in polar:
        tick_label.set_color('#7aaa3d')
    elif str(tick_label)[-5:-2] in positive:
        tick_label.set_color('#b30039')
    elif str(tick_label)[-5:-2] in negative:
        tick_label.set_color('#0c7ace')
    else:
        tick_label.set_color('#ac5cad')

plt.tight_layout();
plt.savefig('/Users/williambrown/Desktop/pdf_figures/hb_conts_250_125_diff.pdf', dpi=300, transparent=True)


In [ ]:
df = pd.DataFrame(newtest.loc[10200])

df.columns = ['result']

df_new = df[df['result'] == True]

df_new


In [ ]:
# nv.color.ColormakerRegistry.add_selection_scheme("my_custom_scheme", [["red", "586"],["red","589"],["red","592"],["red","595"],["red","598"],["red","601"],["red","604"],["red","607"],["red","610"],["red","613"],["red","616"],["red","619"],["red","622"],["red","625"],["red","628"]])

sel = u.select_atoms('resnum 1177 or resnum 593 or resnum 623 or resnum 950 or resnum 1298')
view = nv.show_mdanalysis(sel)
view.add_representation("hyperball")#, color_scheme="my_custom_scheme")
view

In [ ]:
u.trajectory[10200]

with mda.Writer(f'/Users/williambrown/Desktop/250_arg_4cationic.pdb', sel.n_atoms) as f:
    f.write(sel)

In [ ]:
total_50

In [ ]:
thing = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/new_plf_files/PRO_plf.csv', index_col=0)


In [ ]:
len(thing.index.tolist())


In [ ]:
res_50_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/50_1_int_count.csv', index_col=0)
res_50_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/50_2_int_count.csv', index_col=0)
res_50_3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/50_3_int_count.csv', index_col=0)

res_125_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/125_1_int_count.csv', index_col=0)
res_125_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/125_2_int_count.csv', index_col=0)
res_125_3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/125_3_int_count.csv', index_col=0)

res_250_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/250_1_int_count.csv', index_col=0)
res_250_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/250_2_int_count.csv', index_col=0)
res_250_3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/res_interactions/250_3_int_count.csv', index_col=0)

stacked_50 = np.dstack([res_50_1,res_50_2,res_50_3])
stacked_125 = np.dstack([res_125_1,res_125_2,res_125_3])
stacked_250 = np.dstack([res_250_1,res_250_2,res_250_3])

total_50 = pd.DataFrame(np.mean(stacked_50,axis=2), columns=res_50_1.columns, index=res_50_1.index)
err_50 = pd.DataFrame(np.std(stacked_50,axis=2), columns=res_50_1.columns, index=res_50_1.index)

total_125 = pd.DataFrame(np.mean(stacked_125,axis=2), columns=res_125_1.columns, index=res_125_1.index)
err_125 = pd.DataFrame(np.std(stacked_125,axis=2), columns=res_125_1.columns, index=res_125_1.index)

total_250 = pd.DataFrame(np.mean(stacked_250,axis=2), columns=res_250_1.columns, index=res_250_1.index)
err_250 = pd.DataFrame(np.std(stacked_250,axis=2), columns=res_250_1.columns, index=res_250_1.index)

total_50 = total_50.T
total_125 = total_125.T
total_250 = total_250.T

err_50 = err_50.T
err_125 = err_125.T
err_250 = err_250.T

total_50['Pi-Stacking'] += (total_50['EdgeToFace'] +  total_50['FaceToFace'])
total_125['Pi-Stacking'] += (total_125['EdgeToFace'] +  total_125['FaceToFace'])
total_250['Pi-Stacking'] += (total_250['EdgeToFace'] +  total_250['FaceToFace'])

err_50['Pi-Stacking'] += (err_50['EdgeToFace'] + err_50['FaceToFace'])
err_125['Pi-Stacking'] += (err_125['EdgeToFace'] + err_125['FaceToFace'])
err_250['Pi-Stacking'] += (err_250['EdgeToFace'] + err_250['FaceToFace'])

total_50 = total_50.T
total_125 = total_125.T
total_250 = total_250.T

err_50 = err_50.T
err_125 = err_125.T
err_250 = err_250.T

total_50 = total_50.drop(labels=['EdgeToFace','FaceToFace'], axis=0)
total_125 = total_125.drop(labels=['EdgeToFace','FaceToFace'], axis=0)
total_250 = total_250.drop(labels=['EdgeToFace','FaceToFace'], axis=0)

err_50 = err_50.drop(labels=['EdgeToFace','FaceToFace'], axis=0)
err_125 = err_125.drop(labels=['EdgeToFace','FaceToFace'], axis=0)
err_250 = err_250.drop(labels=['EdgeToFace','FaceToFace'], axis=0)

total_50 = total_50.T
total_50['Total'] = total_50['Hydrophobic']+total_50['VdWContact']+total_50['H-Bonding']+total_50['Pi-Cation']+total_50['Electrostatic']+total_50['Pi-Stacking']
total_50 = total_50.sort_values(by='Total')
total_50 = total_50.drop(['Total'], axis=1)
total_50 = total_50.drop(['VdWContact'], axis=1)
total_50 = total_50/1655/5
total_50.insert(loc=0, column='Residue', value=total_50.index)

err_50 = err_50.T
err_50['Total'] = err_50['Hydrophobic']+err_50['VdWContact']+err_50['H-Bonding']+err_50['Pi-Cation']+err_50['Electrostatic']+err_50['Pi-Stacking']
err_50 = err_50.sort_values(by='Total')
err_50 = err_50.drop(['Total'], axis=1)
err_50 = err_50.drop(['VdWContact'], axis=1)
err_50 = err_50/1655/5
err_50.insert(loc=0, column='Residue', value=err_50.index)

total_125 = total_125.T
total_125['Total'] = total_125['Hydrophobic']+total_125['VdWContact']+total_125['H-Bonding']+total_125['Pi-Cation']+total_125['Electrostatic']+total_125['Pi-Stacking']
total_125 = total_125.sort_values(by='Total')
total_125 = total_125.drop(['Total'], axis=1)
total_125 = total_125.drop(['VdWContact'], axis=1)
total_125 = total_125/1523/15
total_125.insert(loc=0, column='Residue', value=total_125.index)

err_125 = err_125.T
err_125['Total'] = err_125['Hydrophobic']+err_125['VdWContact']+err_125['H-Bonding']+err_125['Pi-Cation']+err_125['Electrostatic']+err_125['Pi-Stacking']
err_125 = err_125.sort_values(by='Total')
err_125 = err_125.drop(['Total'], axis=1)
err_125 = err_125.drop(['VdWContact'], axis=1)
err_125 = err_125/1523/15
err_125.insert(loc=0, column='Residue', value=err_125.index)

total_250 = total_250.T
total_250['Total'] = total_250['Hydrophobic']+total_250['VdWContact']+total_250['H-Bonding']+total_250['Pi-Cation']+total_250['Electrostatic']+total_250['Pi-Stacking']
total_250 = total_250.sort_values(by='Total')
total_250 = total_250.drop(['Total'], axis=1)
total_250 = total_250.drop(['VdWContact'], axis=1)
total_250 = total_250/963/30
total_250.insert(loc=0, column='Residue', value=total_250.index)

err_250 = err_250.T
err_250['Total'] = err_250['Hydrophobic']+err_250['VdWContact']+err_250['H-Bonding']+err_250['Pi-Cation']+err_250['Electrostatic']+err_250['Pi-Stacking']
err_250 = err_250.sort_values(by='Total')
err_250 = err_250.drop(['Total'], axis=1)
err_250 = err_250.drop(['VdWContact'], axis=1)
err_250 = err_250/963/30
err_250.insert(loc=0, column='Residue', value=err_250.index)


In [ ]:
plt.subplots(figsize=[5,2])
import matplotlib as mpl
# Create a color palette
palette = dict(zip(['Pi-Stacking','H-Bonding','Pi-Cation','Electrostatic'], ["#e6d800", "#87bc45", "#e60049", "#0d88e6"]))
# Create legend handles manually
handles = [mpl.patches.Patch(color=palette[x], label=x) for x in palette.keys()]
# Create legend
plt.gca().set_axis_off()

plt.legend(handles=handles, ncol=2, fontsize=15)
# Get current axes object and turn off axis

plt.tight_layout();

plt.savefig('/Users/williambrown/Desktop/pdf_figures/novdw_legend_inline.pdf', dpi=300, transparent=True)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

total_50 = total_50.sort_values(by='Residue')
err_50 = err_50.sort_values(by='Residue')
total_50_hphob = total_50[['Residue','Hydrophobic']]
total_50_hphob['Error'] = err_50['Hydrophobic']
total_50_hphob = total_50_hphob.sort_values(by='Hydrophobic')

total_125 = total_125.sort_values(by='Residue')
err_125 = err_125.sort_values(by='Residue')
total_125_hphob = total_125[['Residue','Hydrophobic']]
total_125_hphob['Error'] = err_125['Hydrophobic']
total_125_hphob = total_125_hphob.sort_values(by='Hydrophobic')

total_250 = total_250.sort_values(by='Residue')
err_250 = err_250.sort_values(by='Residue')
total_250_hphob = total_250[['Residue','Hydrophobic']]
total_250_hphob['Error'] = err_250['Hydrophobic']
total_250_hphob = total_250_hphob.sort_values(by='Hydrophobic')

# total_50_strong = total_50[['Residue','Pi-Stacking','H-Bonding','Pi-Cation','Electrostatic']]

# total_125_strong = total_125[['Residue','Pi-Stacking','H-Bonding','Pi-Cation','Electrostatic']]

# total_250_strong = total_250[['Residue','Pi-Stacking','H-Bonding','Pi-Cation','Electrostatic']]


In [ ]:
df = total_50

df_hphob = df[['Residue','Hydrophobic']]

df_strong = pd.DataFrame(index=df['Residue'])
df_strong = df[['Pi-Stacking','H-Bonding','Pi-Cation','Electrostatic']]
df_strong.loc['PRO','H-Bonding'] = df_strong.loc['PRO','H-Bonding'] + df_strong.loc['PRO','Electrostatic']
df_strong.loc['PRO','Electrostatic'] = df_strong.loc['PRO','Electrostatic'] - df_strong.loc['PRO','Electrostatic']
df_strong['total'] = df_strong.sum(axis=1)
df_strong = df_strong.sort_values(by='total')
df_strong = df_strong.iloc[:, :-1]

hphob_palette = dict(zip(['Hydrophobic'],['#bd7ebe']))

palette = dict(zip(['Pi-Stacking','H-Bonding','Pi-Cation','Electrostatic'], ["#e6d800", "#87bc45", "#e60049", "#0d88e6"]))

plt.rcParams["axes.edgecolor"] = "black"

fig, ax = plt.subplots()

df_strong.plot(legend=False,
    kind = 'bar',
    stacked = True,
    mark_right = True,
    grid=False,
    figsize=[7,5],
    color = palette,
    ax=ax)

aromatic = ['PHE', 'TYR', 'TRP']
polar = ['CYS','THR','GLN','ASN','SER']
nonpolar = ['VAL','ILE','ALA','LEU','PRO','GLY','MET']
positive = ['ARG','LYS','HIS']
negative = ['GLU','ASP']

for tick_label in ax.get_xticklabels():
    if str(tick_label)[-5:-2] in aromatic:
        tick_label.set_color('#e3ba14')
    elif str(tick_label)[-5:-2] in polar:
        tick_label.set_color('#7aaa3d')
    elif str(tick_label)[-5:-2] in positive:
        tick_label.set_color('#b30039')
    elif str(tick_label)[-5:-2] in negative:
        tick_label.set_color('#0c7ace')
    else:
        tick_label.set_color('#ac5cad')

# plt.title('Residue Interaction Type Frequency', fontsize=15)
plt.xlabel('Residue', fontsize=15)
plt.ylabel('# of Contacts', fontsize=15)
plt.xticks(fontsize=15, rotation=45)
plt.yticks(fontsize=15)
plt.tight_layout();

plt.savefig('/Users/williambrown/Desktop/pdf_figures/50_stacked_bar.pdf', dpi=300, transparent=True)


In [ ]:
aromatic = ['PHE', 'TYR', 'TRP']
polar = ['CYS','THR','GLN','ASN','SER']
nonpolar = ['VAL','ILE','ALA','LEU','PRO','GLY','MET']
positive = ['ARG','LYS','HIS']
negative = ['GLU','ASP']

classic_colors_50 = ["#e60049" if j in positive else '#e6d800' if j in aromatic else '#87bc45' if j in polar 
                          else '#0d88e6' if j in negative else '#bd7ebe' for j in total_50_hphob['Residue']]

classic_colors_125 = ["#e60049" if j in positive else '#e6d800' if j in aromatic else '#87bc45' if j in polar 
                          else '#0d88e6' if j in negative else '#bd7ebe' for j in total_125_hphob['Residue']]

classic_colors_250 = ["#e60049" if j in positive else '#e6d800' if j in aromatic else '#87bc45' if j in polar 
                          else '#0d88e6' if j in negative else '#bd7ebe' for j in total_250_hphob['Residue']]

fig, ax = plt.subplots(figsize=[7,5])

ax.errorbar(total_50_hphob['Residue'], total_50_hphob['Hydrophobic'], total_50_hphob['Error'], ecolor=classic_colors_50, fmt='o', marker='')
ax.scatter(total_50_hphob['Residue'], total_50_hphob['Hydrophobic'], label='50 mg/mL', color=classic_colors_50, marker='o')#, edgecolor='black')

ax.errorbar(total_125_hphob['Residue'], total_125_hphob['Hydrophobic'], total_125_hphob['Error'], ecolor=classic_colors_125, fmt='o', marker='')
ax.scatter(total_125_hphob['Residue'], total_125_hphob['Hydrophobic'], label='125 mg/mL', color=classic_colors_125, marker='s')#, edgecolor='black')

ax.errorbar(total_250_hphob['Residue'], total_250_hphob['Hydrophobic'], total_250_hphob['Error'], ecolor=classic_colors_250, fmt='o', marker='')
ax.scatter(total_250_hphob['Residue'], total_250_hphob['Hydrophobic'], label='250 mg/mL', color=classic_colors_250, marker='D')#, edgecolor='black')

# plt.grid(visible=False)

for axis in ['top', 'bottom', 'left', 'right']:

    ax.spines[axis].set_linewidth(1.2)  # change width
    ax.spines[axis].set_color('black')    # change color

plt.xticks(fontsize=15)
plt.yticks(fontsize=15)

plt.xticks(rotation=45, fontsize=15)
# plt.ylim(0,1.05)
plt.ylabel('# of Hydrophobic Contacts', fontsize=15, labelpad=-5)
# plt.xlabel('Central Residue')
# plt.title('Hydrophobic', fontsize=15)

import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerBase

list_color  = ["black", "black", "black"]
list_mak    = ["o","s","D"]
list_lab    = ["50 mg/mL", "125 mg/mL", "250 mg/mL"]

# list_color  = ["black", "black"]
# list_mak    = ["s","D"]
# list_lab    = ["125 mg/mL", "250 mg/mL"]

class MarkerHandler(HandlerBase):
    def create_artists(self, legend, tup, xdescent, ydescent,
                        width, height, fontsize, trans):
        return [plt.Line2D([width/2], [height/2.],ls="",
                       marker=tup[1],color=tup[0], transform=trans)]

ax.legend(list(zip(list_color,list_mak)), list_lab,
          bbox_to_anchor=(0.58, 1.023),
          handler_map={tuple:MarkerHandler()}, fontsize=15)

plt.tight_layout();

fig.savefig('/Users/williambrown/Desktop/pdf_figures/hphob_scatter.pdf', dpi=300, transparent=True)


In [ ]:
for dens in [50, 125, 250]:
    
    if dens==50:
        df = total_50
    elif dens==125:
        df = total_125
    else:
        df = total_250
        
    colors = {'Hydrophobic': '#bd7ebe',
              'VdWContact': 'gray',
              'H-Bonding': '#87bc45',
              'Pi-Cation': '#e60049',
              'Electrostatic': '#0d88e6',
              'Pi-Stacking': '#e6d800'}
    
    for res in df.columns:
        
        data = dict(zip(df.index, df[res]))
        
        types = [key for key,value in data.items() if value!=0]
        values = [value for key,value in data.items() if value!=0]
            
        fig,ax = plt.subplots(figsize=[5,5])
        ax = plt.pie(values, colors=[colors[key] for key in types])
        plt.text(-0.3, 1.15, f'{res}', fontsize=40, ma='center')
#         plt.text(-0.33, 1.1, f'{dens} mg/mL', fontsize=15)
        plt.tight_layout();
        plt.savefig(f'/Users/williambrown/Desktop/final_figures/contact_data/res_int_pie_charts/{dens}mgml/{res}_int_pie.png', dpi=500)


In [ ]:
def res_cont_df(dens, res, rep):
    
    if dens == 50:
        num_each = 5
    elif dens == 125:
        num_each = 15
    else:
        num_each = 30
    
    '''function that returns dataframe of counts of 
       each type of interaction for certain residue'''
    
    test = pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/new_plf_files/{res}_plf.csv', header = [0,1,2], index_col=0)
    
    tot_frames = len(test.index)
    series = test.groupby(axis=1, level=2).sum().sum()
    df = pd.DataFrame(series)
    df = df.T
    df = df.divide(tot_frames)
    df = df.divide(num_each)
    
    return df

res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU',
            'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']


In [ ]:
import nglview as nv
import MDAnalysis as mda
from MDAnalysis.analysis import distances
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import numpy as np
import parmed
from parmed import load_file
import datetime
from datetime import datetime

def res_cont_df(dens, res, rep, central_selection):
    
    if dens == 50:
        num_each = 5
    elif dens == 125:
        num_each = 15
    else:
        num_each = 30
    
    '''function that returns dataframe of counts of 
       each type of interaction for certain residue'''
    if res == 'GLY':   
        
        tot_frames = 0

        tot_dict = {'HBAcceptor':0, 'HBDonor':0, 'Hydrophobic':0, 'VdWContact':0,
                    'Cationic':0, 'Anionic':0, 'EdgeToFace':0,'FaceToFace':0, 
                    'PiStacking':0, 'PiCation':0, 'CationPi':0}

        for chunk in pd.read_csv(f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/new_plf_files/{res}_plf.csv', header = [0,1,2], index_col=0, chunksize=5):
                        
            test = chunk[chunk.columns.levels[0][chunk.columns.levels[0].str.fullmatch(central_selection)]]
            
            tot_frames += len(test.index)
            temp_dict = test.groupby(axis=1, level=2).sum().sum().to_dict()
                        
            for i,j in tot_dict.items():
                for k,l in temp_dict.items():
                    if i == k:
                        tot_dict[i] += l
                                        
        tot_dict = {k:[v] for k,v in tot_dict.items()}
        df = pd.DataFrame(tot_dict)
        df = df.T
        df = df.divide(tot_frames)
        df = df.divide(num_each)
                                
    else:
        
        tot_dict = {'HBAcceptor':0, 'HBDonor':0, 'Hydrophobic':0, 'VdWContact':0,
            'Cationic':0, 'Anionic':0, 'EdgeToFace':0,'FaceToFace':0, 
            'PiStacking':0, 'PiCation':0, 'CationPi':0}
    
        test = pd.read_csv(f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/new_plf_files/{res}_plf.csv', header = [0,1,2], index_col=0)

        tot_frames = len(test.index)
        temp_dict = test.groupby(axis=1, level=2).sum().sum().to_dict()
        
        for i,j in tot_dict.items():
            for k,l in temp_dict.items():
                if i == k:
                    tot_dict[i] += l
                                       
        tot_dict = {k:[v] for k,v in tot_dict.items()}
        df = pd.DataFrame(tot_dict)
        df = df.T
        df = df.divide(tot_frames)
        df = df.divide(num_each)
    
    return df

res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU',
            'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']

final_dict = {}

for dens in [125]:

    dens_dict = {}
    
    for rep in [1,2,3]:
        
        top = 'good_output.top'

        if dens == 50:
            num_each = 5
            dcd = '4ms-traj.dcd'
            gly_u = mda.Universe(load_file(f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/{top}'), 
                                    f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/{dcd}')

        elif dens == 125:
            num_each = 15
            dcd = '5ms-traj.dcd'
            gly_u = mda.Universe(load_file(f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/{top}'), 
                                    f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/{dcd}')

        else:
            num_each = 30
            dcd = 'long-traj.dcd'
            dcd2 = 'long-traj2.dcd'

            gly_u = mda.Universe(load_file(f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/{top}'), 
                                    f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/{dcd}',
                                        f'/work/LAS/potoyan-lab/Will/gxg-project/{dens}mgml_100a/replica-{rep}/{dcd2}')

        gly = gly_u.select_atoms('resname GLY')

        terminal_selection = ''
        central_selection = ''

        for resid in gly.residues:
            if len(list(resid.atoms)) != 7:
                terminal_selection += 'GLY' + str(resid.resnum) + '|'
            else:
                central_selection += 'GLY' + str(resid.resnum) + '|'

        terminal_selection = terminal_selection[0:-1]
        central_selection = central_selection[0:-1]

        if rep==1:
            name = 'rep1_df'
        elif rep==2:
            name = 'rep2_df'
        else:
            name = 'rep3_df'
            
        cont_dict = {}

        for res in res_list:
            
            cont_dict[res] = res_cont_df(dens, res, rep, central_selection).T
            
        df = pd.concat(cont_dict.values(), keys=cont_dict.keys())
        df = df.fillna(0)
        df['Pi-Cation'] = df['CationPi'] + df['PiCation']
        df = df.drop(['CationPi', 'PiCation'], axis=1)
        df.rename(columns = {'Pi-Cation':'PiCation'}, inplace = True)
        df = df.reset_index(level=1, drop=True)
        
        dens_dict[name] = df
        
    df_concat = pd.concat((dens_dict['rep1_df'], dens_dict['rep2_df'], dens_dict['rep3_df']))
    avg_df = df_concat.groupby(level=0).mean()
    avg_err = df_concat.groupby(level=0).std()
    
    tot_conts = pd.DataFrame(avg_df.sum(axis=1))
    tot_conts.columns = ['Contacts']
    tot_conts['Error'] = avg_err.sum(axis=1)
    tot_conts = tot_conts.sort_values(by='Contacts', ascending=True)
    
    tot_conts.to_csv(f'./avg_res_conts/{dens}_avg_res_conts.csv')

            
            

In [ ]:
ax = sns.heatmap(avg_conts, cmap="YlOrBr");
ax.set_xlabel('Res-1')
ax.set_ylabel('Res-2')
ax.set_title('Atomistic Cluster Contact Frequencies')
ax.invert_yaxis();
# plt.savefig('/Users/williambrown/Desktop/final_figures/cont_heatmaps/125_atomistic_conts.png', dpi=300, bbox_inches='tight')


In [ ]:
res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU',
            'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']
cont_dict = {}
dens = 125

for res in res_list:
    rep = 1
    cont_dict[res] = res_cont_df(dens, res, rep)

df1 = pd.concat(cont_dict.values(), keys=cont_dict.keys())
df1 = df1.fillna(0)
df1['Pi-Cation'] = df1['CationPi'] + df1['PiCation']
df1 = df1.drop(['CationPi', 'PiCation'], axis=1)
df1.rename(columns = {'Pi-Cation':'PiCation'}, inplace = True)
df1 = df1.reset_index(level=1, drop=True)

In [ ]:
res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU',
            'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']
cont_dict = {}
dens = 125

for res in res_list:
    rep = 2
    cont_dict[res] = res_cont_df(dens, res, rep)

df2 = pd.concat(cont_dict.values(), keys=cont_dict.keys())
df2 = df2.fillna(0)
df2['Pi-Cation'] = df2['CationPi'] + df2['PiCation']
df2 = df2.drop(['CationPi', 'PiCation'], axis=1)
df2.rename(columns = {'Pi-Cation':'PiCation'}, inplace = True)
df2 = df2.reset_index(level=1, drop=True)

In [ ]:
res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU',
            'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']
cont_dict = {}
dens = 125
rep = 3

for res in res_list:
    cont_dict[res] = res_cont_df(dens, res, rep)

df3 = pd.concat(cont_dict.values(), keys=cont_dict.keys())
df3 = df3.fillna(0)
df3['Pi-Cation'] = df3['CationPi'] + df3['PiCation']
df3 = df3.drop(['CationPi', 'PiCation'], axis=1)
df3.rename(columns = {'Pi-Cation':'PiCation'}, inplace = True)
df3 = df3.reset_index(level=1, drop=True)

In [ ]:
dfs = [df1, df2, df3]

df_concat = pd.concat((df1, df2))
avg_df = df_concat.groupby(level=0).mean()
avg_err = df_concat.groupby(level=0).std()


In [ ]:
tot_conts = pd.DataFrame(avg_df.sum(axis=1))

tot_conts.columns = ['Contacts']

tot_conts['Error'] = avg_err.sum(axis=1)


In [ ]:
# u = mda.Universe(load_file('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/output.top'), '/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/5ms-traj.dcd')

cgu = mda.Universe('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-1/cg_traj.pdb')


In [ ]:
arg_dists = pd.DataFrame(columns = ['frame','dist','stdev','total'])
dists = []
stds = []
frames = []

for frame in cgu.trajectory[0:-1:1000]:
    a_group = cgu.select_atoms('resname ARG')
    dist_arr = distances.distance_array(a_group.positions,a_group.positions,box=cgu.dimensions)
    dist_list = []

    for i in dist_arr:
        for j in i:
            if j != 0:
                dist_list.append(j)
                
    frames.append(frame.frame)
    dists.append(np.mean(dist_list))
    stds.append(np.std(dist_list))
    
arg_dists['frame'] = frames
arg_dists['dist'] = dists
arg_dists['stdev'] = stds
# arg_dists['total'] = arg_dists['dist'] * arg_dists['stdev']


dist_sort = arg_dists.sort_values(by='dist')[:15]

# frame_sort = dist_sort.sort_values(by='frame')

# frame_list = list(frame_sort['frame'])

# std_sort = dist_sort.sort_values(by='stdev')[:25]


In [ ]:
# u.trajectory[35800]
protein = cgu.select_atoms('protein')

with mda.Writer(f'/Users/williambrown/Desktop/final_figures/pymol figs/movie_pdbs/250_clustered_frames.pdb', protein.atoms) as f:
    for ts in cgu.trajectory[frame_list]:
        f.write(protein)

In [ ]:
nv.color.ColormakerRegistry.add_selection_scheme("my_custom_scheme", [["red", "131"],["red","133"],["red","135"],["red","137"],["red","139"]])

view = nv.show_mdanalysis(cgu)
view.add_ball_and_stick(color_scheme="my_custom_scheme")
view

In [ ]:
cdict = {'VAL': np.array([0.30126874, 0.44982699, 0.68189158, 1.        ]),
  'ILE': np.array([0.32802768, 0.38985006, 0.65790081, 1.        ]),
  'ALA': np.array([0.32981161, 0.3858516 , 0.65630142, 1.        ]),
  'LEU': np.array([0.32981161, 0.3858516 , 0.65630142, 1.        ]),
  'PRO': np.array([0.36905805, 0.29788543, 0.62111496, 1.        ]),
  'GLY': np.array([0.42614379, 0.16993464, 0.56993464, 1.        ]),
  'PHE': np.array([0.4386313 , 0.14194541, 0.55873895, 1.        ]),
  'CYS': np.array([0.44398308, 0.12995002, 0.55394079, 1.        ]),
  'MET': np.array([0.47609381, 0.0579777 , 0.52515186, 1.        ]),
  'LYS': np.array([0.48322953, 0.04198385, 0.51875433, 1.        ]),
  'HIS': np.array([0.53906959, 0.        , 0.46455978, 1.        ]),
  'THR': np.array([0.54688197, 0.        , 0.45668589, 1.        ]),
  'GLN': np.array([0.56641292, 0.        , 0.43700115, 1.        ]),
  'GLU': np.array([0.65625529, 0.        , 0.34645136, 1.        ]),
  'ASN': np.array([0.66797386, 0.        , 0.33464052, 1.        ]),
  'SER': np.array([0.691411  , 0.        , 0.31101884, 1.        ]),
  'TYR': np.array([0.70312957, 0.        , 0.299208  , 1.        ]),
  'ASP': np.array([0.71484814, 0.        , 0.28739715, 1.        ]),
  'TRP': np.array([0.72266052, 0.        , 0.27952326, 1.        ]),
  'ARG': np.array([1., 0., 0., 1.])}

In [ ]:
import matplotlib

hex_colors = {}

for i,j in cdict.items():
    hex_colors[i] = matplotlib.colors.to_hex(j[0:3])
for i,j in hex_colors.items():
    hex_colors[i] = j.replace('#', '0x')

In [ ]:
res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','HIS','ILE','LEU',
            'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']

with open('/Users/williambrown/Desktop/color_test.txt', 'w') as the_file:
    for i,j in hex_colors.items():
        color_string = 'color ' + j + ', resn ' + i
        the_file.write(color_string + "\n")


In [ ]:
# group_one = ['ARG']
# group_two = ['ASN','ASP','TYR','TRP','SER']
# group_three = ['GLU','GLN','THR','HIS']
# group_four = ['LYS','MET','CYS','PHE','GLY']
# group_five = ['PRO','LEU','ALA','ILE','VAL']

aromatic = ['PHE', 'TYR', 'TRP']
polar = ['CYS','THR','GLN','ASN','SER']
nonpolar = ['VAL','ILE','ALA','LEU','PRO','GLY','MET']
positive = ['ARG','LYS','HIS']
negative = ['GLU','ASP']

# tot_conts = pd.DataFrame(avg_df.sum(axis=1))

# tot_conts.columns = ['Contacts']

# tot_conts['Error'] = avg_err.sum(axis=1)

# tot_conts = tot_conts.sort_values(by='Contacts')

tot_conts = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/contact_data/50_avg_res_conts.csv', index_col=0)

# color_dict = {}
# color_dict = ["red" if j in group_one else 'yellow' if j in group_two else 'seagreen' if j in group_three 
#                           else 'steelblue' if j in group_four else 'rebeccapurple' for j in tot_conts.index]
color_dict = ["#e60049" if j in positive else '#e6d800' if j in aromatic else '#87bc45' if j in polar 
                          else '#0d88e6' if j in negative else '#bd7ebe' for j in tot_conts.index]

# color_list = []
# for res in tot_conts.index:
#     for i,j in cdict.items():
#         if i == res:
#             color_list.append(j)

fig = plt.subplots(figsize=[7,5])
ax = plt.bar(tot_conts.index, tot_conts['Contacts'], yerr=tot_conts['Error'], color=color_dict)
plt.xticks(rotation=90)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.ylim(0,27)
plt.title('Average # Contacts',fontsize=16)
plt.ylabel('Contacts', fontsize=12)
plt.tight_layout()
plt.savefig(f'/Users/williambrown/Desktop/final_figures/plf_results/50mgml/50_total_contacts.png', dpi=300)


In [ ]:
# group_one = ['ARG']
# group_two = ['ASN','ASP','TYR','TRP','SER']
# group_three = ['GLU','GLN','THR','HIS']
# group_four = ['LYS','MET','CYS','PHE','GLY']
# group_five = ['PRO','LEU','ALA','ILE','VAL']

aromatic = ['PHE', 'TYR', 'TRP']
polar = ['CYS','THR','GLN','ASN','SER']
nonpolar = ['VAL','ILE','ALA','LEU','PRO','GLY','MET']
positive = ['ARG','LYS','HIS']
negative = ['GLU','ASP']

int_type_list = ['HBAcceptor', 'HBDonor', 'Hydrophobic', 'VdWContact','Cationic', 
                   'Anionic', 'EdgeToFace', 'FaceToFace', 'PiStacking', 'PiCation']

# int_type_list = ['Cationic']

final_dict = {}
error_dict = {}
for i in int_type_list:
    temp_dict = {}
    temp_err = {}
    
    for j in avg_df.index:
        temp_dict[j] = avg_df[i][j]
        temp_dict = {k:v for k, v in temp_dict.items() if v!=0}
        final_dict[i] = temp_dict
    
    for j in avg_err.index:
        temp_err[j] = avg_err[i][j]
        temp_err = {k:v for k, v in temp_err.items() if v!=0}
        error_dict[i] = temp_err

    sortedDict = sorted(final_dict[i].items(), key=lambda x:x[1])
    new_dict = {}
    for thing in sortedDict:
        new_dict[thing[0]] = thing[1]
        
    df = pd.DataFrame.from_dict(list(new_dict.items()))
    df.columns = ['Residue', 'Contacts']
    df['Error'] = list(temp_err.values())
        
    final_df = pd.DataFrame()
    
    color_dict = {}
#     color_dict[j] = ["red" if j in group_one else 'yellow' if j in group_two else 'seagreen' if j in group_three 
#                           else 'steelblue' if j in group_four else 'rebeccapurple' for j in df['Residue']]
    color_dict[j] = ["red" if j in positive else 'gold' if j in aromatic else 'darkgoldenrod' if j in polar 
                          else 'steelblue' if j in negative else 'grey' for j in df['Residue']]
    if i == 'Anionic':
        plt.bar(df['Residue'][:2], df['Contacts'][:2], yerr=df['Error'][:2], color=color_dict[j])
    else:
        plt.bar(df['Residue'], df['Contacts'], yerr=df['Error'], color=color_dict[j])
    plt.xticks(rotation=90)
    plt.title(f'250 mg/mL - {i} Contacts')
    plt.ylim(0,1.26) 
    plt.ylabel('Average contacts per residue')
    plt.xlabel('Residue')
    plt.tight_layout()
    plt.savefig(f'/Users/williambrown/Desktop/plf_results/250mgml/250_{i}.png', dpi=300)
    plt.clf()

In [ ]:
# def term_cont_filter(dens, rep):

residue_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','HIS','ILE','LEU',
                'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL','GLY']

res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU',
            'LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']

residue_dict = {}

# int_str = 'CationPi|Cationic|HBAcceptor|HBDonor|Anionic|PiStacking|EdgeToEdge|FaceToFace|PiCation'
int_str = 'CationPi|Cationic|HBAcceptor|HBDonor|Anionic|PiCation'
# int_str = 'CationPi|Cationic|HBAcceptor|HBDonor|Anionic|PiStacking|EdgeToEdge|FaceToFace|PiCation'

dens=125
rep=1

for i in res_list:
    residue_dict[i] = []

conts_df = pd.DataFrame()

top = 'good_output.top'

if dens == 50:
    num_each = 5
    dcd = '4ms-traj.dcd'
    gly_u = mda.Universe(load_file(f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/{top}'), 
                            f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/{dcd}')

elif dens == 125:
    num_each = 15
    dcd = '5ms-traj.dcd'
    gly_u = mda.Universe(load_file(f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/{top}'), 
                            f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/{dcd}')

else:
    num_each = 30
    dcd = 'long-traj.dcd'
    dcd2 = 'long-traj2.dcd'

    gly_u = mda.Universe(load_file(f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/{top}'), 
                            f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/{dcd}',
                                f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/{dcd2}')

gly = gly_u.select_atoms('resname GLY')

terminal_selection = ''
central_selection = ''

for resid in gly.residues:
    if len(list(resid.atoms)) != 7:
        terminal_selection += 'GLY' + str(resid.resnum) + '|'
    else:
        central_selection += 'GLY' + str(resid.resnum) + '|'

terminal_selection = terminal_selection[0:-1]
central_selection = central_selection[0:-1]

for chunk in pd.read_csv(f'/Volumes/potoyan-lab/will/gxg-agg-project/{dens}mgml_100a/replica-{rep}/test_plf_files/GLY_plf.csv', header = [0,1,2], index_col=0, chunksize=16):

    residue_df = chunk[chunk.columns.levels[0][chunk.columns.levels[0].str.fullmatch(terminal_selection)]]

    residue_df = residue_df[residue_df.columns[residue_df.columns.get_level_values('interaction').str.fullmatch(int_str)]]
        
#     index_list = []

#     for j,i in enumerate(residue_df):
#         if i[0] != i[1]:
#             index_list.append(j)

#     residue_df = residue_df[residue_df.columns[index_list]]

    for res in res_list:

        if res == 'GLY':

            res_df = residue_df[residue_df.columns[residue_df.columns.get_level_values('protein').str.fullmatch(central_selection)]]
            res_df[res_df.columns[res_df.columns.get_level_values('interaction').str.fullmatch(int_str)]]

        else:

            res_df = residue_df[residue_df.filter(like=f'{res}').columns]
            res_df[res_df.columns[res_df.columns.get_level_values('interaction').str.fullmatch(int_str)]]

        index_list = []

        for j,i in enumerate(res_df):
            resid_1 = int(i[0][3:])
            resid_2 = int(i[1][3:])
            if resid_1 != resid_2 and resid_1 != resid_2+1 and resid_1 != resid_2-1:
                index_list.append(j)
            else:
                continue

        res_df = res_df.iloc[:,index_list]

        frames = residue_df.shape[0]

        residue_dict[res].append(res_df.values.sum() / frames)

for i,j in residue_dict.items():

    conts_df.loc['GLY', i] = np.mean(j) / num_each

# conts_df.to_csv(f'term_conts_{dens}_{rep}.csv')

#     return conts_df

conts_df


In [ ]:
term_conts50_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/50mgml_100a/replica-1/term_hb_conts.csv', index_col=0)
term_conts50_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/50mgml_100a/replica-2/term_hb_conts.csv', index_col=0)
term_conts50_3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/50mgml_100a/replica-3/term_hb_conts.csv', index_col=0)

term_conts50_1 = term_conts50_1 / 2
term_conts50_2 = term_conts50_2 / 2
term_conts50_3 = term_conts50_3 / 2

term_conts125_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-1/term_hb_conts.csv', index_col=0)
term_conts125_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-2/term_hb_conts.csv', index_col=0)
term_conts125_3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/125mgml_100a/replica-3/term_hb_conts.csv', index_col=0)

term_conts125_1 = term_conts125_1 / 2
term_conts125_2 = term_conts125_2 / 2
term_conts125_3 = term_conts125_3 / 2

term_conts250_1 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-1/term_hb_conts.csv', index_col=0)
term_conts250_2 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-2/term_hb_conts.csv', index_col=0)
term_conts250_3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-agg-project/250mgml_100a/replica-3/term_hb_conts.csv', index_col=0)

term_conts250_1 = term_conts250_1 / 2
term_conts250_2 = term_conts250_2 / 2
term_conts250_3 = term_conts250_3 / 2

term_50 = (term_conts50_1 + term_conts50_2 + term_conts50_3) / 3
term_125 = (term_conts125_1 + term_conts125_2 + term_conts125_3) / 3
term_250 = (term_conts250_1 + term_conts250_2 + term_conts250_3) / 3

In [ ]:
term_50.to_csv()

In [ ]:
test = term_250 - term_125

In [ ]:
test


In [ ]:
# ax = sns.heatmap(term_50, cmap='YlOrRd', square=True, norm=LogNorm(), cbar_kws={"orientation": "horizontal", "pad": 0.05, 'ticks':[0.08,0.2,0.3,0.6,1.0,2.0,4.0]})

test = test[['ASP','GLU','ARG','LYS','HIS','TRP','TYR','PHE','ASN','GLN','SER','THR','CYS','PRO','VAL','MET','ILE','LEU','GLY']]

ax = sns.heatmap(test, cmap=cmap, square=True, vmin=-0.32, vmax= 0.32, cbar=False)

# sns.set(font_scale=0.9)

cmap = sns.diverging_palette(260, 20, as_cmap=True, s=80, l=50)

pc = ax.pcolormesh(test, norm=colors.TwoSlopeNorm(vmin=-0.32, vcenter=0.0, vmax=0.32), cmap=cmap)

cbar = plt.colorbar(pc, ax=ax, orientation='horizontal', pad=0.03, ticks=[-0.3,-0.2,-0.1,0.0,0.1,0.2,0.3])
# colorbar = ax.collections[0].colorbar

# tick_size = 10

cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=13)

# colorbar.tick_params(labelsize=tick_size, length=4, width=1)

# colorbar.set_ticklabels(['-0.2','-0.1','0.0','0.1','0.2','0.3'])
                                                                                 
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
# plt.title('Terminal Hydrogen Bonding', fontsize=15)
plt.yticks(fontsize=13, rotation=0)
plt.xticks(fontsize=13, rotation=45)

for tick_label in ax.get_xticklabels():
    if str(tick_label)[-5:-2] in aromatic:
        tick_label.set_color('#e3ba14')
    elif str(tick_label)[-5:-2] in polar:
        tick_label.set_color('#7aaa3d')
    elif str(tick_label)[-5:-2] in positive:
        tick_label.set_color('#b30039')
    elif str(tick_label)[-5:-2] in negative:
        tick_label.set_color('#0c7ace')
    else:
        tick_label.set_color('#ac5cad')

plt.tight_layout();
plt.savefig('/Users/williambrown/Desktop/pdf_figures/term_hbconts_250125_diff_0.3.pdf', dpi=300, transparent=True)


# 

# 